<a href="https://colab.research.google.com/github/Rojoniaina-Rasoafarihy/Evo2_project/blob/main/embedding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
!pip install torch==2.7.1 --index-url https://download.pytorch.org/whl/cu128
!pip install flash-attn==2.8.0.post2 --no-build-isolation
!pip install evo2

In [ ]:
import os

WORK_DIR = "/content"
CACHE_DIR = "/content/hf"

os.makedirs(CACHE_DIR, exist_ok=True)
os.environ["HF_HOME"] = f"{CACHE_DIR}"

In [ ]:
import os
import torch
import numpy as np
from Bio import SeqIO
from tqdm import tqdm
from evo2 import Evo2

In [ ]:
!nvidia-smi
print("CUDA available:", torch.cuda.is_available())

In [ ]:
#  Load Evo2 model
evo2_model = Evo2('evo2_7b')
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
layer_name = 'blocks.28'

In [ ]:
#  Download input data from GCS

!mkdir -p /home/jupyter/data
!gsutil -m cp -r gs://rojo_project/vfdb/vfdb_context /home/jupyter/data/

INPUT_DIR = "/home/jupyter/data/vfdb_context"
print("Number of FASTA files:", len([f for f in os.listdir(INPUT_DIR) if f.endswith(".fasta")]))


In [ ]:
#  Output directory for .npz files
OUTPUT_DIR = "/home/jupyter/data/evo2_embeddings_npz"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
#  Function: get mean‑pooled embedding for a single sequence
def get_sequence_embedding(sequence, model, layer_name, device):
    tokens = model.tokenizer.tokenize(sequence)
    input_ids = torch.tensor(tokens, dtype=torch.int).unsqueeze(0).to(device)

    with torch.no_grad():
        _, embeddings = model(
            input_ids,
            return_embeddings=True,
            layer_names=[layer_name]
        )
    # embeddings[layer_name] shape: (1, L, D) -> squeeze to (L, D)
    per_base = embeddings[layer_name].squeeze(0).cpu()
    # mean over length → (D,)
    return per_base.float().mean(dim=0).numpy()


In [ ]:
#  Process one genome FASTA: embed all windows, aggregate,
#  save as .npz
def process_genome_fasta(fasta_path, output_npz_path, model, layer_name, device):
    window_vectors = []

    for record in SeqIO.parse(fasta_path, "fasta"):
        seq = str(record.seq)
        if len(seq) == 0:
            continue

        try:
            vec = get_sequence_embedding(seq, model, layer_name, device)
            window_vectors.append(vec)
        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"OOM → skipping {record.id}")
                torch.cuda.empty_cache()
                continue
            else:
                raise e

    if not window_vectors:
        print(f"No valid windows in {fasta_path}, skipping")
        return

    # Aggregate: mean + max concatenation
    arr = np.stack(window_vectors, axis=0)   # (n_windows, D)
    mean_vec = arr.mean(axis=0)
    max_vec = arr.max(axis=0)
    genome_embedding = np.concatenate([mean_vec, max_vec])   # (2*D,)

    # Save as compressed numpy
    np.savez_compressed(output_npz_path, embedding=genome_embedding)


In [ ]:
#  Main loop over all FASTA files
fasta_files = [f for f in os.listdir(INPUT_DIR) if f.endswith(".fasta")]

for fasta_file in tqdm(fasta_files):
    genome_stem = fasta_file.replace(".fasta", "")
    input_path = os.path.join(INPUT_DIR, fasta_file)
    output_path = os.path.join(OUTPUT_DIR, f"{genome_stem}.npz")

    if os.path.exists(output_path):
        continue

    process_genome_fasta(input_path, output_path, evo2_model, layer_name, device)


#  Upload results back to GCS
!gsutil -m cp -r /home/jupyter/data/evo2_embeddings_npz gs://rojo_project/

In [ ]:
#verification
sample_npz = os.path.join(OUTPUT_DIR, os.listdir(OUTPUT_DIR)[0])
data = np.load(sample_npz)
print(f"Sample file: {sample_npz}")
print(f"Embedding shape: {data['embedding'].shape}")   # Should be (2*D,)
print(f"Number of genomes processed: {len(os.listdir(OUTPUT_DIR))}")

Sample file: /home/jupyter/data/evo2_embeddings_npz/GCA_040673135.1.npz
Embedding shape: (8192,)
Number of genomes processed: 29920
